# Scraping Data from Wikipedia

In [37]:
# Calling in BeautifulSoup and Requests
from bs4 import BeautifulSoup
import requests

In [38]:
# Getting our urls
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_rail_transport_network_size'
# url_1 = 'https://en.wikipedia.org/wiki/List_of_largest_companies_in_the_United_States_by_revenue'

# Getting our pages
page = requests.get(url)
# page_1 = requests.get(url_1)

# Extract text from the page using BeautifulSoup
soup = BeautifulSoup(page.text, 'html')
# soup_1 = BeautifulSoup(page.text, 'html')

In [39]:
print(soup)
# print(soup_1)

<!DOCTYPE html>

<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 vector-feature-night-mode-enabled skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available" dir="ltr" lang="en">
<head>
<meta charset="utf-8"/>
<title>List of countries by rail transport network size - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limit

In [40]:
# Getting the needed table(1st scraped table)
table_info = soup.find_all('table', class_ = 'wikitable')[0]

In [41]:
# Getting all the table titles
table_titles = table_info.find_all('th')

In [42]:
# Setting the titles needed
desired_titles = ['Country/Territory', 'Total', 'Electrified', "% of the totalelectrified",
    'Area (km2)', 'Population', 'Historical peaklength (km)',
    'Nationalized or private[a]', 'Data year']

# Looping through table titles to filter out the ones needed
table_titles_text = [title.text.strip() for title in table_titles if title.text.strip() in desired_titles]
print(table_titles_text)

['Country/Territory', '% of the totalelectrified', 'Historical peaklength (km)', 'Nationalized or private[a]', 'Data year', 'Total', 'Electrified', 'Area (km2)', 'Population']


In [43]:
# Importing pandas for putting the data into a dataframe
import pandas as pd

In [44]:
# Getting the titles from the filtered list
df = pd.DataFrame(columns = table_titles_text)

# Rearranging the titles
df = df[['Country/Territory', 'Total', 'Electrified', '% of the totalelectrified',
'Area (km2)', 'Population', 'Historical peaklength (km)',
'Nationalized or private[a]', 'Data year']]

# Checking if the titles put in and rearranged
df

,Country/Territory,Total,Electrified,% of the totalelectrified,Area (km2),Population,Historical peaklength (km),Nationalized or private[a],Data year


In [45]:
# Editing the titles
df.columns = ['Country/Territory', 'Total (km)', 'Electrified (km)', '% of the total electrified',
              'Area (km^2)', 'Population(per km)', 'Historical peak length (km)',
              'Nationalized or private', 'Data year']
# Checking the edited titles
df

,Country/Territory,Total (km),Electrified (km),% of the total electrified,Area (km^2),Population(per km),Historical peak length (km),Nationalized or private,Data year


In [46]:
# Getting the table data in rows from the scraped table
column_data = table_info.find_all('tr')

In [47]:
import re

# Looping through the rows of the table data
for row in column_data[2:]: # First two columns are empty
   # Getting the every row data with td tags 
   row_data = row.find_all('td')[0:9] # Data until the 9th column is relevant
   # Getting the text from the row without the newline in a list
   # Removing the brackets, spaces, and newlines from the text
   each_row_data = [re.sub(r"\[.*?\]", "", data.text).replace('\n', '').strip() for data in row_data]

   # print(each_row_data)
   
   # Matching each row's column length to the dataframe's column length to prevent mismatched columns
   if (len(each_row_data) == len(df.columns)):
        # Getting the length of the dataframe and adding each row to the dataframe, based on the index
        length = len(df)
        df.loc[length] = each_row_data

df

,Country/Territory,Total (km),Electrified (km),% of the total electrified,Area (km^2),Population(per km),Historical peak length (km),Nationalized or private,Data year
0,United States,"220,044","2,011",0.91%,44.69,"1,522","428,180 (1917)","Track ownership and freight mostly private, pa...",2019
1,China,"159,000","119,000",74.84%,60.61,"8,865","159,000 (2023)",Nationalized,2023
2,Russia,"105,000","54,054",51.48%,162.84,"1,367","105,000(2023)",Nationalized,2022
3,India,"68,584","64,244",97.64%,47.93,"21,038","68,584 (2023)",Nationalized with minimal private operators,2024
4,Canada,"49,422",129,0.20%,214.48,674,"69,636 (1940)",Freight - private Passenger - public,2017
...,...,...,...,...,...,...,...,...,...
146,Nauru,3.9,0,0.00%,4.20,"2,000",,,2001
147,Monaco,1.7,1.7,100.00%,1.18,"20,588",3.5 (1868–1958),Nationalized (operated by France),2024
148,Lesotho,1.6,0,0.00%,"10,118.33","723,667",,,1995
149,Vatican City,0.3,0,0.00%,1.47,"3,333",0.3 (since 1934),Nationalized (operated by Italy),2024


In [ ]:
# Exporting the dataframe to a CSV file
df.to_csv(r'C:\Users\Shriyansh Singh\Desktop\Bootcamps\Data Analyst Bootcamp\FreeCodeCamp\Python Jupyter Files\Python Projects\Data_Scraped_File\Train_Networks.csv', index = False)